# RawGrad — Gradient Descent

This notebook demonstrates gradient descent using the RawGrad neural-network framework.

We will:

- Build a small neural network
- Perform forward propagation
- Calculate the loss
- Perform backpropagation
- Update model parameters using gradient descent
- Track the loss during training
- Verify that the model learns

The goal is to understand how gradients calculated during backpropagation are converted into parameter updates.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from rawgrad.layers import Dense
from rawgrad.activations import ReLU, Softmax
from rawgrad.model import Sequential
from rawgrad.classification_losses import CategoricalCrossEntropy
from rawgrad.optimizers import SGD

In [2]:
np.random.seed(42)

X = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 1.0, 0.5],
    [0.0, 1.0, 2.0],
    [3.0, 2.0, 1.0],
])

labels = np.array([0, 1, 2, 1])

targets = np.zeros((len(labels), 3))
targets[np.arange(len(labels)), labels] = 1.0

print("Input shape:", X.shape)
print("Target shape:", targets.shape)
print("Labels:", labels)

Input shape: (4, 3)
Target shape: (4, 3)
Labels: [0 1 2 1]


In [3]:
model = Sequential([
    Dense(3, 8, initialization="xavier"),
    ReLU(),
    Dense(8, 3, initialization="xavier"),
    Softmax(),
])

loss_function = CategoricalCrossEntropy()

optimizer = SGD(learning_rate=0.1)

print("Model created.")
print("Optimizer: SGD")
print("Learning rate:", 0.1)

Model created.
Optimizer: SGD
Learning rate: 0.1


In [4]:
predictions = model.forward(X)

initial_loss = loss_function.forward(predictions, targets)

print("Initial loss:", initial_loss)

Initial loss: 0.8227435211326628


In [5]:
loss_gradient = loss_function.backward()

model.backward(loss_gradient)

optimizer.step(model)

print("One gradient descent step completed.")

One gradient descent step completed.


In [6]:
updated_predictions = model.forward(X)

updated_loss = loss_function.forward(
    updated_predictions,
    targets,
)

print("Loss before update:", initial_loss)
print("Loss after update :", updated_loss)
print("Loss change       :", updated_loss - initial_loss)

Loss before update: 0.8227435211326628
Loss after update : 0.6632456640818751
Loss change       : -0.15949785705078767


In [7]:
np.random.seed(42)

model = Sequential([
    Dense(3, 8, initialization="xavier"),
    ReLU(),
    Dense(8, 3, initialization="xavier"),
    Softmax(),
])

loss_function = CategoricalCrossEntropy()
optimizer = SGD(learning_rate=0.1)

loss_history = []

epochs = 300

for epoch in range(epochs):
    predictions = model.forward(X)

    loss = loss_function.forward(
        predictions,
        targets,
    )

    loss_history.append(loss)

    gradient = loss_function.backward()

    model.backward(gradient)

    optimizer.step(model)

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch + 1:3d} | "
            f"Loss: {loss:.6f}"
        )

Epoch  50 | Loss: 0.333233
Epoch 100 | Loss: 0.285084
Epoch 150 | Loss: 0.194589
Epoch 200 | Loss: 0.118200
Epoch 250 | Loss: 0.070845
Epoch 300 | Loss: 0.045463


In [8]:
print("Initial loss:", loss_history[0])
print("Final loss:  ", loss_history[-1])
print("Loss reduction:", loss_history[0] - loss_history[-1])

Initial loss: 0.8227435211326628
Final loss:   0.04546328170996974
Loss reduction: 0.7772802394226931


In [9]:
final_predictions = model.forward(X)

predicted_classes = np.argmax(
    final_predictions,
    axis=1,
)

accuracy = np.mean(
    predicted_classes == labels
)

print("Actual labels:   ", labels)
print("Predicted classes:", predicted_classes)
print(f"Training accuracy: {accuracy * 100:.2f}%")

Actual labels:    [0 1 2 1]
Predicted classes: [0 1 2 1]
Training accuracy: 100.00%


In [10]:
assert np.isfinite(loss_history[0])
assert np.isfinite(loss_history[-1])

assert loss_history[-1] < loss_history[0]

assert final_predictions.shape == (4, 3)
assert np.all(np.isfinite(final_predictions))

assert np.allclose(
    np.sum(final_predictions, axis=1),
    1.0,
)

print("GRADIENT DESCENT: PASS")

GRADIENT DESCENT: PASS
